In [0]:
# Databricks notebook source
# Camada Bronze - Arquitetura Medalhão
# Objetivo:
# 1) criar catálogo/schema da camada bronze;
# 2) ingerir os CSVs do dataset Olist para tabelas Delta;
# 3) registrar metadados de ingestão;
# 4) ingerir a cotação do dólar via API do Banco Central.

from datetime import datetime
import requests

from pyspark.sql import DataFrame
from pyspark.sql import functions as F


In [0]:
# Definição de parâmetros
dbutils.widgets.text("catalogo", "ecommerce", "Catálogo")
dbutils.widgets.text("schema_bronze", "bronze", "Schema Bronze")
dbutils.widgets.text("volume_path", "/Volumes/ecommerce/bronze/raw_data", "Caminho do Volume")

catalogo = dbutils.widgets.get("catalogo").strip()
schema_bronze = dbutils.widgets.get("schema_bronze").strip()
volume_path = dbutils.widgets.get("volume_path").rstrip("/")

In [0]:
# Criação da estrutura física da camada Bronze.
# O volume raw_data é o diretório esperado para armazenar os CSVs brutos no Lakehouse.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_bronze}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalogo}.{schema_bronze}.raw_data")

print(f"Catálogo: {catalogo}")
print(f"Schema Bronze: {schema_bronze}")
print(f"Volume esperado: {volume_path}")


Catálogo: ecommerce
Schema Bronze: bronze
Volume esperado: /Volumes/ecommerce/bronze/raw_data


In [0]:
# Função de ingestão CSV -> Bronze.
# Regra de negócio da Bronze:
# - não transformamos o dado de negócio;
# - apenas lemos o arquivo bruto, adicionamos metadados de rastreabilidade
#   e persistimos em Delta para consumo nas próximas camadas.
def ingest_csv(nome_arquivo: str, nome_tabela: str) -> None:
    landing_path = f"{volume_path}/{nome_arquivo}"

    try:
        df: DataFrame = spark.read.csv(landing_path, header=True, inferSchema=True)

        # Validação mínima para evitar a criação de tabela vazia por erro de caminho/arquivo.
        if df.limit(1).count() == 0:
            raise ValueError(f"O arquivo {landing_path} está vazio ou não pôde ser lido.")

        # Metadados de linhagem e auditoria.
        df = (
            df.withColumn("timestamp_ingestion", F.current_timestamp())
        )

        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(f"{catalogo}.{schema_bronze}.{nome_tabela}")
        )

        print(f"✅ Tabela {catalogo}.{schema_bronze}.{nome_tabela} criada com sucesso.")

    except Exception as e:
        print(f"❌ Erro ao processar {nome_tabela}: {str(e)}")
        raise


In [0]:
# Mapeamento dos arquivos brutos para as tabelas da camada Bronze
# conforme especificado no enunciado da atividade.
mapeamento_arquivos_tabela_bronze = {
    "olist_customers_dataset": "tb_customers",
    "olist_geolocation_dataset": "tb_geolocalizacao",
    "olist_order_items_dataset": "tb_order_items",
    "olist_order_payments_dataset": "tb_order_payments",
    "olist_order_reviews_dataset": "tb_order_reviews",
    "olist_orders_dataset": "tb_orders",
    "olist_products_dataset": "tb_products",
    "olist_sellers_dataset": "tb_sellers",
    "product_category_name_translation": "tb_product_category_name_translation"
}

for arquivo, tabela in mapeamento_arquivos_tabela_bronze.items():
    ingest_csv(f"{arquivo}.csv", tabela)


✅ Tabela ecommerce.bronze.tb_customers criada com sucesso.
✅ Tabela ecommerce.bronze.tb_geolocalizacao criada com sucesso.
✅ Tabela ecommerce.bronze.tb_order_items criada com sucesso.
✅ Tabela ecommerce.bronze.tb_order_payments criada com sucesso.
✅ Tabela ecommerce.bronze.tb_order_reviews criada com sucesso.
✅ Tabela ecommerce.bronze.tb_orders criada com sucesso.
✅ Tabela ecommerce.bronze.tb_products criada com sucesso.
✅ Tabela ecommerce.bronze.tb_sellers criada com sucesso.
✅ Tabela ecommerce.bronze.tb_product_category_name_translation criada com sucesso.


In [0]:
import requests
from datetime import timedelta

# tabelas
tabela_orders = f"{catalogo}.{schema_bronze}.tb_orders"
tabela_cotacao_dolar = f"{catalogo}.{schema_bronze}.tb_cotacao_dolar"

# pega intervalo de datas dos pedidos
datas = (
    spark.table(tabela_orders)
    .select(F.to_date("order_purchase_timestamp").alias("data_pedido"))
    .agg(
        F.min("data_pedido").alias("data_inicio"),
        F.max("data_pedido").alias("data_fim")
    )
    .collect()[0]
)

data_inicio = datas["data_inicio"]
data_inicio = data_inicio - timedelta(days=4)
data_fim = datas["data_fim"]

if data_inicio is None or data_fim is None:
    raise ValueError("Não foi possível obter datas da tb_orders")

# formata para MM-DD-AAAA (formato exigido pela API)
data_inicio_formatada = data_inicio.strftime("%m-%d-%Y")
data_fim_formatada = data_fim.strftime("%m-%d-%Y")

# URL da API
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&"
    f"@dataFinalCotacao='{data_fim_formatada}'&"
    "$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    cotacoes = response.json().get("value", [])

    if not cotacoes:
        print("⚠️ Nenhuma cotação retornada (fim de semana/feriado).")
    else:
        df_cotacao = (
            spark.createDataFrame(cotacoes)
            .withColumn("timestamp_ingestion", F.current_timestamp())
        )

        (
            df_cotacao.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_cotacao_dolar)
        )

        print(f"✅ Tabela {tabela_cotacao_dolar} criada com sucesso")
        print(f"Período: {data_inicio_formatada} até {data_fim_formatada}")

except Exception as e:
    print(f"❌ Erro: {str(e)}")
    raise

✅ Tabela ecommerce.bronze.tb_cotacao_dolar criada com sucesso
Período: 08-31-2016 até 10-17-2018
